In [ ]:
import os, subprocess, torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
tok = secrets.get_secret("GH_TOKEN")
os.environ["KAGGLE_API_TOKEN"] = secrets.get_secret("KAGGLE_API_TOKEN")  # needed for --checkpoint-dataset push

if not os.path.isdir("/kaggle/working/boltU"):
    subprocess.run(["git", "clone", f"https://{tok}@github.com/dhmizu95/boltU.git",
                    "/kaggle/working/boltU"], check=True)
os.chdir("/kaggle/working/boltU")
subprocess.run(["git", "pull", "--ff-only"], check=True)

subprocess.run("pip install -q tiktoken pyyaml datasets gradio kaggle".split(), check=True)
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
assert free_gb > 12, f"only {free_gb:.1f} GB free in /kaggle/working"
print(f"{free_gb:.1f} GB free in /kaggle/working")

In [ ]:
# Resume automatically if a prior checkpoint is attached as a dataset (multi-session runs);
# a fresh Tier A run has none, so this starts from step 0.
import os, subprocess
resume_flag = ["--resume"] if os.path.exists("/kaggle/input/boltu-checkpoints/ckpt_latest.pt") else []
subprocess.run([
    "python", "src/train.py",
    "--config", "configs/base.yaml",
    *resume_flag,
    "--checkpoint-dataset", "delowerhossainmizu/boltu-checkpoints",
], check=True)